# Distribution Shift, Feedback Loops, and Governance Lab


## Purpose

This lab simulates distribution shift and connects monitoring to governance.

Learning goals:

- Compare reference and current feature distributions.
- Compute a simple drift metric.
- Document governance questions triggered by drift.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

reference = rng.normal(loc=0.0, scale=1.0, size=5000)
current_mild = rng.normal(loc=0.30, scale=1.05, size=5000)
current_severe = rng.normal(loc=0.90, scale=1.25, size=5000)

def population_stability_index(reference_values, current_values, bins=10):
    reference_counts, bin_edges = np.histogram(reference_values, bins=bins)
    current_counts, _ = np.histogram(current_values, bins=bin_edges)

    reference_pct = reference_counts / max(reference_counts.sum(), 1)
    current_pct = current_counts / max(current_counts.sum(), 1)

    epsilon = 1e-6

    return np.sum(
        (current_pct - reference_pct)
        * np.log((current_pct + epsilon) / (reference_pct + epsilon))
    )

drift = pd.DataFrame([
    {
        "comparison": "reference_vs_mild_shift",
        "psi": population_stability_index(reference, current_mild),
    },
    {
        "comparison": "reference_vs_severe_shift",
        "psi": population_stability_index(reference, current_severe),
    },
])

drift

In [ ]:
governance_checklist = pd.DataFrame([
    {"layer": "data", "question": "What changed in the data-generating process?"},
    {"layer": "model", "question": "Does current performance still match validation evidence?"},
    {"layer": "decision", "question": "Are outputs still appropriate for operational use?"},
    {"layer": "monitoring", "question": "Who reviews drift alerts and how quickly?"},
    {"layer": "governance", "question": "Should the model be retrained, constrained, paused, or retired?"},
])

governance_checklist

## Interpretation

Drift metrics are signals, not governance by themselves. Responsible systems define review procedures, escalation rules, and human accountability before deployment.